# Notebook to demo GWR modeling.

In [ ]:
import warnings

# warnings.simplefilter(action='ignore', category=FutureWarning)
# warnings.filterwarnings("ignore")

In [ ]:
from collections import OrderedDict
from typing import Tuple

import numpy as np
import pandas as pd
import pyspark.sql.functions as F
from itertools import product
from statsmodels.regression.linear_model import WLS

import geofunctions as S

In [ ]:
warnings.simplefilter(action='ignore', category=FutureWarning)
spark.version

In [ ]:
xmin, ymin, xmax, ymax = (
    35.46417853366222,
    33.841706023612225,
    35.57065336261585,
    33.906762501363865,
)

xdel = xmax - xmin
ydel = ymax - ymin

cell1 = min(xdel, ydel) / 10.0
cell2 = cell1 * 0.5

### Create dataframe with geom, dep var and ind vars.

In [ ]:
df = (
    spark
    .range(1000)
    .drop("id")
    .withColumn("x", F.rand() * xdel + xmin)
    .withColumn("y", F.rand() * ydel + ymin)
    # Define dep attribute
    .withColumn("y0", F.rand() * 100)
    # define ind attributes
    .withColumn("x1", F.rand() * 100)
    .withColumn("x2", F.rand() * 100)
)

In [ ]:
df = (df
      .withColumn("q", S.st_xtoq("x", cell1))
      .withColumn("r", S.st_ytor("y", cell1))
     )

### Define neighborhood range.

In [ ]:
data = list(product(range(-1, 2), range(-1, 2)))
prod = spark.createDataFrame(data, "i int,j int")

In [ ]:
def gwr(qr:Tuple[float,float], df:pd.DataFrame) -> pd.DataFrame:
    q, r = qr
    # calculate center
    x0 = q * cell1 + cell2
    y0 = r * cell1 + cell2
    dx = df.x - x0
    dy = df.y - y0
    # Get weights based on distance.
    w = np.clip(1.0 - np.sqrt(dx * dx + dy * dy) / cell1, 0.0, 1.0)
    if np.min(w) < np.max(w):
        model = WLS(df["y0"], df[["x1", "x2"]], w)
        res = model.fit()
        data = [
            ("q", [q]),
            ("r", [r]),
            ("n", [len(df)]),
            ("p0", [res.params[0]]),
            ("p1", [res.params[1]]),
            ("r2", [res.rsquared]),
        ]
    else:
        data = [
            ("q", [q]),
            ("r", [r]),
            ("n", [0]),
            ("p0", [0.0]),
            ("p1", [0.0]),
            ("r2", [-1.0]),
        ]
    return pd.DataFrame(OrderedDict(data))

In [ ]:
with pd.option_context('mode.chained_assignment', None):
    (
        df
        .crossJoin(F.broadcast(prod))
        .withColumn("q", F.col("q") + F.col("i"))
        .withColumn("r", F.col("r") + F.col("j"))
        .drop("i", "j")
        .groupBy("q", "r")
        .applyInPandas(
            gwr, schema="q long, r long, n int, p0 double, p1 double, r2 double"
        )
        .show()
    )